# Rawaj Outreach & Follow-Up Agent

This is the original team notebook, extended for the updated flow. The implementation remains in the adjacent Python files.

التجربة الفعلية: **Ashi Sushi** — **fatimah.alamri.official@gmail.com**. شغّلي الخلايا بالترتيب داخل المشروع المحدّث. أبقينا نفس الأقسام وطريقة استدعاء الـruntime.
استخدمي Kernel جديدًا عند بدء الجلسة. البريد وLLM وStrategy حقيقيون؛ الإرسال الأول يتطلب قراءة المسودة ثم كتابة APPROVE.
بيانات Research وQualification محفوظة من المشروع؛ لا نعيد البحث عنها. قاعدة التجربة تحفظ نسخة Ashi فقط ولا تغيّر قاعدة الفريق.

## 1. Workflow at a glance

`Research + Qualification → personalized draft → review → Human approve/reject with reason → real email with Yes/No`

`Yes → verified Strategy handoff → actual Strategy → automatic email 2 with trial + credentials + link`

`No → DO_NOT_CONTACT`. `No response → automatic personalized follow-up (demo: 2 minutes; production: 7 days)`.

`First activation → 30-day trial → feedback request near expiry → saved feedback`.
Human approval applies only to the first outreach. A rejected draft is regenerated and shown for approval again.

In [ ]:
from __future__ import annotations
import json, subprocess, sys, os, sqlite3, secrets, time, threading
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT/'Rawaj_Project_SDA_Fixed'/'agents').exists():
    PROJECT_ROOT = PROJECT_ROOT/'Rawaj_Project_SDA_Fixed'
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT/'agents').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT/'agents').exists(), 'افتحي النوتبوك داخل مجلد المشروع المحدّث'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## 2. Configuration and real integrations

القيم التالية للتجربة على بريدك فقط. غيّري ENVIRONMENT إلى production للانتظار 7 أيام.
يُقرأ `.env` ثم `.env.live` إن وجد، دون عرض المفاتيح. إذا كانت ناقصة، يطلبها إدخالًا مخفيًا.
مفاتيح الجلسة تُحفظ محليًا داخل work؛ لا تشاركيه. افتحي أزرار البريد من الكمبيوتر نفسه وأبقي Kernel شغّالًا.

In [ ]:
assert 'database.database' not in sys.modules, 'ابدئي بـ Restart Kernel قبل إعداد جلسة جديدة'
from dotenv import dotenv_values
from getpass import getpass
for filename in ['.env', '.env.live']:
    os.environ.update({k:v for k,v in dotenv_values(PROJECT_ROOT/filename).items() if v})
ENVIRONMENT = 'demo'
TEST_EMAIL = 'fatimah.alamri.official@gmail.com'
PORT = 8000
BASE_URL = f'http://127.0.0.1:{PORT}'
PUBLIC_BASE_URL = BASE_URL  # الهاتف يحتاج رابط HTTPS عام موجّه لنفس السيرفر قبل توليد البريد
# Use a clean demo session so an orphaned draft from an earlier kernel cannot
# block this run while the original session remains available for inspection.
SESSION = PROJECT_ROOT/'work'/'ashi_notebook_clean'
SESSION.mkdir(parents=True, exist_ok=True)
secret_path = SESSION/'session-secrets.json'
if not secret_path.exists():
    assert not (SESSION/'live.db').exists(), 'قاعدة الجلسة موجودة لكن مفاتيحها مفقودة؛ استعيدي session-secrets.json الأصلي ولا تنشئي كلمة مرور بديلة'
    secret_path.write_text(json.dumps({k:secrets.token_urlsafe(48) for k in
        ['ACCOUNT_SECRET','BUTTON_SIGNING_SECRET','ADMIN_API_TOKEN']}), encoding='utf-8')
os.environ.update(json.loads(secret_path.read_text(encoding='utf-8')))
os.environ.update(DATABASE_URL='sqlite:///'+(SESSION/'live.db').as_posix(),
    LANGGRAPH_CHECKPOINT_PATH=str(SESSION/'checkpoints.sqlite'),
    STRATEGY_HANDOFF_OUTBOX=str(SESSION/'handoffs'/'inbox'),
    RAWAJ_ENV=ENVIRONMENT, FOLLOW_UP_DELAY_MINUTES='2' if ENVIRONMENT=='demo' else '10080',
    PUBLIC_BASE_URL=PUBLIC_BASE_URL, ALLOW_LOCAL_BUTTON_DEMO='true',
    DASHBOARD_URL='http://127.0.0.1:8501', RAWAJ_API_URL=BASE_URL, RAWAJ_DEMO_LOGIN='false', LANGSMITH_TRACING='false')
for key in ['OPENAI_API_KEY']:
    if not os.environ.get(key): os.environ[key]=getpass(key+': ').strip()
model = (os.environ.get('OPENAI_MODEL') or os.environ.get('OPENAI_DECISION_MODEL') or
    input('اسم نموذج OpenAI المتاح لحساب API: ').strip())
assert model
for key in ['OPENAI_MODEL','OPENAI_GENERATION_MODEL','OPENAI_DECISION_MODEL','OPENAI_REVIEW_MODEL']:
    os.environ[key]=model
provider = os.environ.get('EMAIL_PROVIDER') or 'smtp'
os.environ['EMAIL_PROVIDER']=provider
assert provider.lower() in {'smtp','resend'}
required = ['FROM_EMAIL','RESEND_API_KEY'] if provider.lower()=='resend' else ['FROM_EMAIL','SMTP_HOST','SMTP_USERNAME','SMTP_PASSWORD']
for key in required:
    if not os.environ.get(key): os.environ[key]=getpass(key+': ').strip()
from agents.outreach_followup_agent.config import reload_settings
settings = reload_settings()
settings.require_email()
settings.require_button_endpoint()
print('OpenAI model:', settings.openai_decision_model)
print('Provider:', settings.email_provider, '| Follow-up minutes:', settings.follow_up_delay_minutes)

## 3. Prompt, schemas, tools, and LangGraph state

The master rules are split by stage so the LLM sees only the relevant instructions at each decision, generation, and review step. The tools are real LangChain `@tool` definitions, while LangGraph owns the conditional routing and approval interruption/resume behavior.

In [ ]:
from agents.outreach_followup_agent.prompt import CORE_AGENT_CONTRACT
from agents.outreach_followup_agent.tools import OUTREACH_FOLLOWUP_TOOLS
from agents.outreach_followup_agent.schemas import OutreachGraphState

print(CORE_AGENT_CONTRACT)
print('\nAvailable tools:')
for tool in OUTREACH_FOLLOWUP_TOOLS:
    print(f'- {tool.name}: {tool.description.splitlines()[0]}')
print('\nLangGraph state keys:', list(OutreachGraphState.__annotations__.keys()))

## 4. Real Research + Qualification handoff data

نستخدم Ashi Sushi من قاعدة المشروع. لا نستخدم mock data في الرحلة. نحدّد IDs من بيانات المطعم بدل افتراض أرقام ثابتة.

In [ ]:
with sqlite3.connect(PROJECT_ROOT/'rawaj.db') as source:
    row=source.execute('SELECT id,name FROM restaurants WHERE lower(name)=?', ('ashi sushi',)).fetchone()
    assert row, 'Ashi Sushi غير موجود في قاعدة المشروع'
    RESTAURANT_ID=row[0]
    run=source.execute("SELECT id,research_run_id FROM qualification_runs WHERE restaurant_id=? AND lower(qualification)='qualified' ORDER BY id DESC LIMIT 1", (RESTAURANT_ID,)).fetchone()
    assert run, 'المطعم يحتاج Qualification مؤهلة محفوظة'
    QUALIFICATION_RUN_ID, RESEARCH_RUN_ID=run
print(row[1], '| Research:', RESEARCH_RUN_ID, '| Qualification:', QUALIFICATION_RUN_ID)

## 5. Offline verification (optional)

هذا القسم الأصلي اختياري للتحقق من الكود فقط. باقي النوتبوك رحلة فعلية، ولا تعتمد عليه لتحديد وصول البريد.

In [ ]:
RUN_OFFLINE_CHECKS = False
if RUN_OFFLINE_CHECKS:
    test_run=subprocess.run([sys.executable,'-m','pytest','tests/test_requested_lifecycle.py','-q'],
        cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(test_run.stdout)
    assert test_run.returncode==0, test_run.stderr
else:
    print('Skipped optional local tests; continuing to real runtime.')

## 6. Create the real runtime and linked dashboard

قاعدة واحدة ومفاتيح ثابتة للنوتبوك وOutreach وStrategy والدخول. يبدأ الخادم الكامل والواجهة الأصلية معًا، والعامل التلقائي يبقى متوقفًا حتى قسم 9.
إذا كانت جلستك الحالية تحتوي حسابًا وخطة، تُستكمل كما هي. لا نعيد إرسال رسائلها ولا نغيّر كلمة المرور.
ثبّتي متطلبات المشروع و `rawaj_front/requirements.txt` في بيئة النوتبوك. يجب إيقاف سيرفر النوتبوك القديم قبل التشغيل؛ لا يشغّل هذا النوتبوك خادمًا جزئيًا بديلًا.

In [ ]:
from database.database import Base, engine, SessionLocal
from database import models
assert hasattr(models,'ClientTrial') and hasattr(models,'ClientFeedbackRecord'), 'استخدمي ملفات الباكند المحدّثة المرفقة؛ النوتبوك وحده لا يضيف موديلات قاعدة البيانات'
Base.metadata.create_all(engine)
with sqlite3.connect(PROJECT_ROOT/'rawaj.db') as source, sqlite3.connect(SESSION/'live.db') as dest:
    source.row_factory=sqlite3.Row
    for table, identifier in [('restaurants',RESTAURANT_ID),('research_runs',RESEARCH_RUN_ID),('qualification_runs',QUALIFICATION_RUN_ID)]:
        if dest.execute('SELECT 1 FROM '+table+' WHERE id=?',(identifier,)).fetchone(): continue
        row=dict(source.execute('SELECT * FROM '+table+' WHERE id=?',(identifier,)).fetchone())
        if table=='restaurants': row['email']=TEST_EMAIL
        columns={r[1] for r in dest.execute('PRAGMA table_info('+table+')')}
        row={k:v for k,v in row.items() if k in columns}
        dest.execute('INSERT INTO '+table+' ('+','.join(row)+') VALUES ('+','.join('?' for _ in row)+')',list(row.values()))

from scripts.linked_workspace import configure
configure(SESSION)
from database.outreach_repository import OutreachRepository
repository=OutreachRepository()
import requests, hashlib
headers={'X-Admin-Token':os.environ['ADMIN_API_TOKEN']}
def api(method,path,**kwargs):
    response=requests.request(method,BASE_URL+path,headers=headers,timeout=600,**kwargs)
    response.raise_for_status()
    return response.json()
expected_session=hashlib.sha256(str(SESSION.resolve()).encode()).hexdigest()[:16]
try:
    status=api('GET','/api/local/session')
except requests.ConnectionError:
    log=open(SESSION/'launcher.log','a',encoding='utf-8')
    linked_process=subprocess.Popen([sys.executable,'-m','scripts.linked_workspace','--session',str(SESSION)],
        cwd=PROJECT_ROOT,env=os.environ.copy(),stdout=log,stderr=log)
    for _ in range(120):
        if linked_process.poll() is not None: raise RuntimeError('تعذر التشغيل؛ راجعي launcher.log داخل الجلسة')
        try:
            status=api('GET','/api/local/session')
            break
        except requests.ConnectionError: time.sleep(.5)
    else: raise TimeoutError('الخادم لم يبدأ')
except requests.HTTPError:
    raise RuntimeError('المنفذ يستخدم سيرفرًا قديمًا أو جلسة مختلفة. أوقفي السيرفر القديم ثم أعيدي الخلية.') from None
assert status['session_id']==expected_session,'الخادم متصل بجلسة مختلفة؛ لا تكملي قبل تصحيحها'
print('Same session verified. Dashboard:',status['dashboard_url'])

## 7. Start initial outreach from the team pipeline

ينشئ أول draft من نتائج Research وQualification ويقف عند Human Approval. لا يرسل بريدًا في هذا القسم.

In [ ]:
from agents.outreach_followup_agent.guardrails import safe_preview
from database.models import Restaurant
with SessionLocal() as db:
    restaurant=db.get(Restaurant,RESTAURANT_ID)
    assert restaurant.name.lower()=='ashi sushi' and restaurant.email==TEST_EMAIL
    print('Restaurant:',restaurant.name, '| Recipient:',restaurant.email)

In [ ]:
research_handoff=repository.load_research_handoff(restaurant_id=RESTAURANT_ID,research_run_id=RESEARCH_RUN_ID)
qualification_handoff=repository.load_qualification_handoff(restaurant_id=RESTAURANT_ID,
    research_run_id=RESEARCH_RUN_ID,qualification_run_id=QUALIFICATION_RUN_ID)
print('Research status:',research_handoff['analysis_status'])
print('Qualification:',qualification_handoff['qualification'])

In [ ]:
journey=api('GET',f'/api/local/journey/{RESTAURANT_ID}')
if journey['messages']:
    print('Existing journey retained; no new first email generated.')
    print(journey)
else:
    print(api('POST','/api/local/outreach/start',json={'restaurant_id':RESTAURANT_ID,
        'research_run_id':RESEARCH_RUN_ID,'qualification_run_id':QUALIFICATION_RUN_ID}))

## 8. Inspect and approve the exact email

The reviewer must inspect the stored rendered draft and use the binding values in the LangGraph interruption payload. Approval applies to that exact message ID, revision, and content hash; editing a draft requires a new review.

In [ ]:
pending=[p for p in api('GET','/api/approvals') if p['restaurant_id']==RESTAURANT_ID]
draft=pending[0] if pending else None
if draft:
    print('Subject:',draft['subject'])
    print('Draft:',safe_preview(draft['body']),sep=chr(10))
    reviewed_message_id=draft['message_id']
else:
    print('لا توجد مسودة معلقة. إذا أرسلتِ سابقًا انتقلي إلى الحالة أو افتحي الداشبورد، ولا تعيدي إرسال أول رسالة.')

In [ ]:
# The full API and existing Streamlit dashboard were started together in section 6.
status=api('GET','/api/local/session')
assert status['session_id']==expected_session
ui=requests.get('http://127.0.0.1:8501/_stcore/health',timeout=10)
ui.raise_for_status()
print('Dashboard ready:',status['dashboard_url'])
print('Old /live/strategy email links redirect here automatically.')

In [ ]:
if draft is None:
    print('No pending initial draft; nothing sent.')
else:
    choice=input('APPROVE للإرسال أو REJECT لإعادة المسودة: ').strip().upper()
    assert choice in {'APPROVE','REJECT'}
    reason=input('سبب الرفض: ').strip() if choice=='REJECT' else None
    assert choice!='REJECT' or reason,'سبب الرفض مطلوب'
    decision=api('POST',f'/api/approvals/{reviewed_message_id}/decision',json={
        'decision':'APPROVED' if choice=='APPROVE' else 'REJECTED','reviewer_id':'fatimah','note':reason})
    print(decision)
    if choice=='REJECT': print('أعيدي عرض المسودة ثم هذه الخلية لمراجعة النسخة الجديدة.')

## 9. Verified Interested response and Strategy Agent handoff

الزر الحقيقي من البريد → تسجيل Yes → طلب Strategy بمعرّف ثابت → حفظ الخطة → إشارة READY → رسالة الرابط والحساب تلقائيًا.
الخلية التالية تبدأ العامل داخل الخادم الكامل نفسه. عند الفشل يظهر stage/failure ويتوقف العامل بدل تكرار استدعاءات مدفوعة بلا نهاية. بعد حل السبب شغّلي الخلية مجددًا؛ الخطة المحفوظة يُعاد استخدامها.
عند No تُوقف المتابعة. عدم الرد يُتابع حسب المدة المحددة. الحساب والخطة والدخول جميعها من قاعدة الجلسة نفسها.

In [ ]:
print(api('POST','/api/local/pipeline/start'))

In [ ]:
journey=api('GET',f'/api/local/journey/{RESTAURANT_ID}')
print(json.dumps(journey,ensure_ascii=False,indent=2))
print('saved_strategy_ids = خطة محفوظة. STRATEGY_READY_NOTIFICATION/SENT = قبول مزود البريد للرسالة الثانية.')
print('افتحي Gmail للتأكد من الوصول الفعلي. الرابط: http://127.0.0.1:8501/strategy')

In [ ]:
from IPython.display import display, Markdown
display(Markdown('[افتحي واجهة Rawaj والخطة الفعلية](http://127.0.0.1:8501/strategy)'))

## 10. Activation and the 30-day free trial

افتحي الداشبورد وأدخلي نفس اسم المستخدم وكلمة المرور الموجودين في الرسالة الثانية. بعد الدخول اختاري Monthly Strategy من القائمة لعرض الخطة الفعلية.
أول دخول صحيح يبدأ 30 يومًا؛ إعادة الدخول لا تعيد العداد. الخلية التالية اختيارية للتحقق من الحساب ومدة التجربة عبر نفس API، ولا تطبع كلمة المرور.

In [ ]:
import requests
username=getpass('rawaj_username من الرسالة الثانية: ').strip()
password=getpass('rawaj_password: ')
login=requests.post(BASE_URL+'/api/auth/login',json={'username':username,'password':password},timeout=30)
del password
login.raise_for_status()
owner_headers={'Authorization':'Bearer '+login.json()['access_token']}
trial=requests.get(BASE_URL+'/api/client/trial',headers=owner_headers,timeout=30)
trial.raise_for_status()
print(trial.json())

## 11. Feedback near trial expiry

العامل يطلب feedback عند موعده قرب نهاية التجربة (اليوم 27). لا نغيّر الوقت أو نقول إن 30 يومًا مرت. تتطلب المتابعة بقاء العامل شغّالًا، أو تشغيل العامل الدائم في المشروع لاحقًا.
يمكنك تجربة حفظ رأيك الفعلي الآن بعد الدخول. هذا إرسال feedback حقيقي إلى جدول ClientFeedbackRecord، وليس رسالة بريد.

In [ ]:
from database.models import ClientTrial, ClientFeedbackRecord
with SessionLocal() as db:
    trial_row=db.get(ClientTrial,RESTAURANT_ID)
    if trial_row:
        print('Activated:',trial_row.activated_at,'| Expires:',trial_row.expires_at)
        print('Feedback due:',trial_row.feedback_due_at)
    else: print('Not activated yet.')

In [ ]:
feedback_text=input('ملاحظتك الفعلية (اتركيه فارغًا لتخطي الحفظ): ').strip()
if feedback_text:
    rating=int(input('التقييم من 1 إلى 5: '))
    response=requests.post(BASE_URL+'/api/client/feedback',headers=owner_headers,
        json={'message':feedback_text,'rating':rating},timeout=30)
    response.raise_for_status()
    feedback_id=response.json()['feedback_id']
    with SessionLocal() as db:
        saved_feedback=db.get(ClientFeedbackRecord,feedback_id)
        assert saved_feedback and saved_feedback.restaurant_id==RESTAURANT_ID
    print('Feedback saved and verified:',feedback_id)

## 12. Stop the automatic pipeline

إغلاق النوتبوك لا يحتاج إيقاف الداشبورد. الخلية التالية توقف المتابعة الآلية فقط، وتبقي رابط الدخول صالحًا طالما الخادم يعمل.
لا تفتحي سيرفرًا ثانيًا على نفس المنفذ. لتشغيل نفس الجلسة لاحقًا استخدمي قسم 6. لا تحذفي قاعدة الجلسة أو مفاتيحها.

In [ ]:
STOP_PIPELINE = False
if STOP_PIPELINE: print(api('POST','/api/local/pipeline/stop'))

## Guardrails reference for the presentation
The executable policies are in `agents/outreach_followup_agent/guardrails.py`.
- `SecretProtector` and `safe_preview`: hide usernames, passwords and tokens in model inputs and previews.
- `validate_proposal`: block generated links, credential text, known unsafe promises and unsupported references.
- `validate_provenance`, `strategy_errors`, `require_matching_recipient`: validate the handoff and recipient.
- `OutreachGuardrails`: eligibility, opt-out, timing, limits and exact approval binding.
- `credential_delivery_error`: allow account details only in the strategy-ready email body.
Use `safe_preview(value)` before printing operational objects. Private delivery bodies and checkpoints are not encrypted by redaction.
